# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

This section audits two findings from the FlyRank research paper using the same validation and leakage questions applied to my own model.

Source: `docs/flyrank-seo-research-march-2026.pdf` — "The State of AI-Driven SEO, March 2026."

### Finding chosen #1 — "Growth Prediction Coefficients" (ML Appendix, p.29)

**What the paper claims:** a logistic regression reaches 71% holdout accuracy separating growing from declining pages, with `content_age_days`, `days_since_last_update`, and `days_visible` as the strongest coefficients.

**My methodology question (constructive):** the paper's methodology describes an 80/20 split for the ML models, but does not state whether pages were grouped by brand before splitting. The portfolio spans multiple brands, so if pages from the same brand appear in both train and test, the model could partly learn brand-level momentum rather than a page-level pattern that transfers to a completely unseen brand.

**Question:** was the 80/20 split grouped by brand, or was it row-level random? The paper does not make this distinction explicit, and the two designs provide different guarantees about generalization.

### Finding chosen #2 — "AI Model Performance" (Finding #10, p.16)

**What the paper claims:** an age-controlled comparison of OpenAI- and Gemini-authored pages shows that the two providers trade the lead across different age tiers, so no single provider consistently wins. The finding is framed as **NUANCED**, rather than **CONFIRMED**.

**My methodology question (constructive):** age is controlled, which is useful, but the outcome being compared is `health_score`, a FlyRank composite based partly on visibility metrics such as impressions and average position. Since topic mix, publication timing, and promotion effort can also affect visibility, the question is whether age control alone is enough to support the comparison, or whether additional matching or controls would strengthen it.

The paper's **NUANCED** framing is appropriate because the result does not establish that authorship itself caused the observed performance difference.

### Why I chose these two

Both findings are useful examples of evidence that is reasonably strong but still benefits from careful methodological framing. The goal of this audit is not to "disprove" the paper, but to ask whether the validation design supports the strength of the claim.

In [1]:
# This section is a structured record of the two methodology audits.

paper_audit = [
    {
        "finding": "Growth Prediction Coefficients (logistic regression, 71% holdout accuracy)",
        "source_page": 29,
        "label_origin": (
            "trend_direction (growth vs decline), computed from 30d-vs-prev-30d "
            "impression change"
        ),
        "methodology_question": (
            "Was the 80/20 holdout split grouped by brand, or row-level random? "
            "If pages from the same brand appear in both train and test, the model "
            "could partly learn brand-level momentum rather than a page-level pattern "
            "that transfers to an unseen brand."
        ),
        "verdict": (
            "Directionally useful, but the split design is not disclosed clearly "
            "enough to confirm generalization to an unseen brand."
        ),
    },
    {
        "finding": "AI Model Performance — OpenAI vs Gemini, age-controlled cohorts",
        "source_page": 16,
        "label_origin": (
            "health score, a composite incorporating visibility and engagement "
            "metrics including impressions and average position"
        ),
        "methodology_question": (
            "Age is controlled, but health score is a downstream composite of "
            "visibility metrics. Does age control alone rule out topic-mix or "
            "promotion-effort confounds?"
        ),
        "verdict": (
            "The paper already labels this finding NUANCED rather than CONFIRMED, "
            "which is appropriately cautious given the composite outcome and "
            "possible remaining confounds."
        ),
    },
]

for item in paper_audit:
    print(f"- {item['finding']}")
    print(f"    label origin: {item['label_origin']}")
    print(f"    methodology question: {item['methodology_question']}")
    print(f"    verdict: {item['verdict']}")
    print()

- Growth Prediction Coefficients (logistic regression, 71% holdout accuracy)
    label origin: trend_direction (growth vs decline), computed from 30d-vs-prev-30d impression change
    methodology question: Was the 80/20 holdout split grouped by brand, or row-level random? If pages from the same brand appear in both train and test, the model could partly learn brand-level momentum rather than a page-level pattern that transfers to an unseen brand.
    verdict: Directionally useful, but the split design is not disclosed clearly enough to confirm generalization to an unseen brand.

- AI Model Performance — OpenAI vs Gemini, age-controlled cohorts
    label origin: health score, a composite incorporating visibility and engagement metrics including impressions and average position
    methodology question: Age is controlled, but health score is a downstream composite of visibility metrics. Does age control alone rule out topic-mix or promotion-effort confounds?
    verdict: The paper alread

## 2. My model under an honest split

Week 5 already used a client-grouped holdout as the honest validation design. 
For this audit, the "before" case is a naive row-level random split, while the 
"after" case holds out entire clients.

Both splits use the same Random Forest model, preprocessing, features, target, 
and random seed. The only thing that changes is the validation design.

This makes the difference in performance attributable to the split design rather 
than to a different model or feature set.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import sklearn

SEED = 42
np.random.seed(SEED)

print(
    f"scikit-learn {sklearn.__version__} | "
    f"numpy {np.__version__} | "
    f"pandas {pd.__version__}"
)

df_raw = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "log_impressions_90d",
    "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "has_word_count",
]

CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier",
    "position_tier",
]

TARGET = "is_declining_label"

# Same preparation as Week 5
df = df_raw.copy()

df = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].copy()

df = df.drop_duplicates(
    subset=["content_id"]
).reset_index(drop=True)

df[TARGET] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

# Feature engineering
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

df["has_word_count"] = df["word_count"].notna().astype(int)

# Clean numeric features
for col in NUMERIC_FEATURES:
    df[col] = (
        pd.to_numeric(df[col], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

# Clean categorical features
for col in CATEGORICAL_FEATURES:
    df[col] = (
        df[col]
        .fillna("unknown")
        .astype(str)
        .replace({"nan": "unknown", "": "unknown"})
    )

print(
    f"Prepared: {len(df):,} rows | "
    f"label rate: {df[TARGET].mean():.1%} declining"
)


def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({
        "y": y_true.values,
        "score": scores
    })

    top = (
        frame
        .sort_values("score", ascending=False)
        .head(k)
    )

    return float(top["y"].mean()) if len(top) else 0.0


preprocessor = ColumnTransformer([
    (
        "num",
        StandardScaler(),
        NUMERIC_FEATURES
    ),
    (
        "cat",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ),
        CATEGORICAL_FEATURES
    ),
])


def fit_eval(train_df, test_df, label):

    X_train = train_df[
        NUMERIC_FEATURES + CATEGORICAL_FEATURES
    ]

    y_train = train_df[TARGET]

    X_test = test_df[
        NUMERIC_FEATURES + CATEGORICAL_FEATURES
    ]

    y_test = test_df[TARGET]

    rf = Pipeline([
        (
            "pre",
            preprocessor
        ),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=10,
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1
            )
        ),
    ])

    rf.fit(X_train, y_train)

    proba = rf.predict_proba(X_test)[:, 1]

    p20 = precision_at_k(
        y_test,
        proba,
        20
    )

    p50 = precision_at_k(
        y_test,
        proba,
        50
    )

    auc = roc_auc_score(
        y_test,
        proba
    )

    print(
        f"{label:28s} "
        f"n_train={len(train_df):6,} "
        f"n_test={len(test_df):6,} "
        f"P@20={p20:.3f}  "
        f"P@50={p50:.3f}  "
        f"ROC-AUC={auc:.3f}"
    )

    return {
        "label": label,
        "n_train": len(train_df),
        "n_test": len(test_df),
        "P@20": p20,
        "P@50": p50,
        "ROC-AUC": auc
    }


print()
print("=== BEFORE vs AFTER: split design comparison ===\n")


# BEFORE:
# Naive row-level random split
train_rand, test_rand = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df[TARGET]
)

before = fit_eval(
    train_rand,
    test_rand,
    "BEFORE: random split"
)


# AFTER:
# Client-grouped split
clients = sorted(
    df["client_id"].unique()
)

test_clients = set(
    clients[::5]
)

train_grp = df[
    ~df["client_id"].isin(test_clients)
].copy()

test_grp = df[
    df["client_id"].isin(test_clients)
].copy()

after = fit_eval(
    train_grp,
    test_grp,
    "AFTER: client-grouped split"
)


print()

gap_p50 = (
    before["P@50"] -
    after["P@50"]
)

gap_auc = (
    before["ROC-AUC"] -
    after["ROC-AUC"]
)

print(
    f"Gap from switching to the honest split: "
    f"P@50 drops by {gap_p50:.3f}, "
    f"ROC-AUC drops by {gap_auc:.3f}."
)

print(
    "That gap IS the finding -- it's how much of the random-split score "
    "was the model partially recognizing a client's environment rather "
    "than a page-level pattern."
)

scikit-learn 1.9.0 | numpy 2.5.0 | pandas 3.0.3
Prepared: 30,000 rows | label rate: 54.2% declining

=== BEFORE vs AFTER: split design comparison ===

BEFORE: random split         n_train=24,000 n_test= 6,000 P@20=0.950  P@50=0.900  ROC-AUC=0.764
AFTER: client-grouped split  n_train=24,490 n_test= 5,510 P@20=0.400  P@50=0.580  ROC-AUC=0.657

Gap from switching to the honest split: P@50 drops by 0.320, ROC-AUC drops by 0.107.
That gap IS the finding -- it's how much of the random-split score was the model partially recognizing a client's environment rather than a page-level pattern.


## 3. Leakage audit

The model's label is derived from `trend_direction`, and `trend_pct` is directly related to that label. I therefore deliberately inject `trend_pct` into the feature set as a known leakage test.

If the validation harness is working correctly, this deliberately leaky model should achieve near-perfect performance. The real model must then exclude `trend_pct` and `trend_direction` from its features.

In [3]:
# Deliberately inject a leaky feature to test whether the validation
# harness can detect obvious label leakage.

NUMERIC_LEAKY = NUMERIC_FEATURES + ["trend_pct"]

# Add trend_pct to the prepared dataframe
df["trend_pct"] = pd.to_numeric(
    df_raw.loc[df.index, "trend_pct"],
    errors="coerce"
).fillna(0)

preprocessor_leaky = ColumnTransformer([
    (
        "num",
        StandardScaler(),
        NUMERIC_LEAKY
    ),
    (
        "cat",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ),
        CATEGORICAL_FEATURES
    ),
])

X_train_leaky = train_grp[
    NUMERIC_LEAKY + CATEGORICAL_FEATURES
]

y_train_leaky = train_grp[TARGET]

X_test_leaky = test_grp[
    NUMERIC_LEAKY + CATEGORICAL_FEATURES
]

y_test_leaky = test_grp[TARGET]

rf_leaky = Pipeline([
    (
        "pre",
        preprocessor_leaky
    ),
    (
        "clf",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        )
    ),
])

rf_leaky.fit(
    X_train_leaky,
    y_train_leaky
)

proba_leaky = rf_leaky.predict_proba(
    X_test_leaky
)[:, 1]

p50_leaky = precision_at_k(
    y_test_leaky,
    proba_leaky,
    50
)

auc_leaky = roc_auc_score(
    y_test_leaky,
    proba_leaky
)

print(
    "WITH trend_pct injected "
    "(label-derived, deliberately leaky):"
)

print(
    f"  P@50={p50_leaky:.3f}  "
    f"ROC-AUC={auc_leaky:.3f}"
)

print(
    "WITHOUT it "
    "(the real, honest AFTER model above):"
)

print(
    f"  P@50={after['P@50']:.3f}  "
    f"ROC-AUC={after['ROC-AUC']:.3f}"
)

print()

if p50_leaky > 0.95 and auc_leaky > 0.95:
    print(
        "CONFIRMED: injecting the leaky column pushes "
        "the score toward a perfect 1.0."
    )
    print(
        "This confirms that the test harness correctly "
        "detects obvious leakage."
    )
else:
    print(
        "WARNING: leaky feature did not push the score "
        "near 1.0 -- investigate the harness."
    )

WITH trend_pct injected (label-derived, deliberately leaky):
  P@50=1.000  ROC-AUC=1.000
WITHOUT it (the real, honest AFTER model above):
  P@50=0.580  ROC-AUC=0.657

CONFIRMED: injecting the leaky column pushes the score toward a perfect 1.0.
This confirms that the test harness correctly detects obvious leakage.


### 3.2 Full leakage attack checklist

The deliberate leakage test above confirms that the validation harness reacts strongly to an obviously label-derived feature. I now apply the full checklist to the actual, non-leaky feature set used by the capstone model.

In [4]:
# Full leakage / validation attack checklist
real_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES

checklist = {}

# 1. Timeline
# All features are available from the 90-day snapshot and are not derived
# from the trend_direction label.
checklist["timeline_drawn"] = True


# 2. No label-derived features
label_derived = {
    "trend_direction",
    "trend_pct"
}

checklist["no_label_derived_features"] = (
    label_derived.isdisjoint(set(real_features))
)


# 3. No existing product flags / system scores
product_flags = {
    "health_score",
    "optimization_flags",
    "flag_count"
}

checklist["no_product_flags"] = (
    product_flags.isdisjoint(set(real_features))
)


# 4. Grouped split
# Confirm that no client appears in both training and test.
checklist["grouped_split_used"] = True

assert set(
    train_grp["client_id"]
).isdisjoint(
    set(test_grp["client_id"])
)


# 5. Base rate reported
checklist["base_rate_reported"] = True

base_rate = float(
    y_test_leaky.mean()
)


# 6. Feature importance sanity check
# Week 5 permutation importance showed traffic / engagement
# features among the strongest signals rather than label-derived columns.
checklist["top_features_sanity_checked"] = True


# 7. IDs never used as features
id_cols = {
    "content_id",
    "client_id"
}

checklist["ids_not_features"] = (
    id_cols.isdisjoint(set(real_features))
)


# 8. Metrics are calculated on held-out data
checklist["out_of_fold_metrics"] = True


print("=== Attack checklist — final feature set ===")

for k, v in checklist.items():
    print(
        f"  [{'x' if v else ' '}] {k}"
    )


print(
    f"\nBase rate (declining) on honest test set: "
    f"{base_rate:.1%}"
)

print(
    f"Honest model P@50 = {after['P@50']:.3f} "
    f"vs base rate {base_rate:.1%} "
    f"-> {after['P@50'] - base_rate:+.3f} "
    f"lift over naive always-flag ranking."
)


assert all(
    checklist.values()
), "One or more leakage guards failed -- stop and fix before trusting the model."


print("\nAll checklist items pass.")

=== Attack checklist — final feature set ===
  [x] timeline_drawn
  [x] no_label_derived_features
  [x] no_product_flags
  [x] grouped_split_used
  [x] base_rate_reported
  [x] top_features_sanity_checked
  [x] ids_not_features
  [x] out_of_fold_metrics

Base rate (declining) on honest test set: 48.4%
Honest model P@50 = 0.580 vs base rate 48.4% -> +0.096 lift over naive always-flag ranking.

All checklist items pass.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [5]:
boldest_original = (
    "Both learned models improve on the rule baseline at precision@20 and precision@50 "
    "on the client-held-out test set."
)

boldest_rewrite = (
    "On a client-grouped holdout -- pages from clients never seen during training -- the "
    "Random Forest is OBSERVED to outperform the Week-4 rule baseline at precision@50. "
    "This is a DIRECTIONAL, decision-support signal for which pages a reviewer should look "
    "at first, not a guarantee for any single new client. The naive random split produced "
    "a substantially higher P@50, showing why client-grouped validation is important."
)

print("ORIGINAL (bold claim):")
print(f"  {boldest_original}")
print()
print("REWRITTEN (safe, split-aware claim):")
print(f"  {boldest_rewrite}")

ORIGINAL (bold claim):
  Both learned models improve on the rule baseline at precision@20 and precision@50 on the client-held-out test set.

REWRITTEN (safe, split-aware claim):
  On a client-grouped holdout -- pages from clients never seen during training -- the Random Forest is OBSERVED to outperform the Week-4 rule baseline at precision@50. This is a DIRECTIONAL, decision-support signal for which pages a reviewer should look at first, not a guarantee for any single new client. The naive random split produced a substantially higher P@50, showing why client-grouped validation is important.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.